In [ ]:

import re
import copy

# --- 1. DATA STRUCTURES & PARSING ---

class Term:
    def __init__(self, name, args=None):
        self.name = name
        self.args = args if args is not None else [] # Empty args = variable or constant

    def is_variable(self):
        # Lowercase single-word identifiers with no args are variables (e.g., x, y)
        return not self.args and self.name[0].islower()

    def __repr__(self):
        if not self.args:
            return self.name
        return f"{self.name}({', '.join(map(str, self.args))})"

    def __eq__(self, other):
        return isinstance(other, Term) and self.name == other.name and self.args == other.args

    def __hash__(self):
        return hash((self.name, tuple(self.args)))


class Literal:
    def __init__(self, predicate, args, negated=False):
        self.predicate = predicate
        self.args = args
        self.negated = negated

    def negate(self):
        return Literal(self.predicate, copy.deepcopy(self.args), not self.negated)

    def __repr__(self):
        neg = "~" if self.negated else ""
        return f"{neg}{self.predicate}({', '.join(map(str, self.args))})"

    def __eq__(self, other):
        return (isinstance(other, Literal) and self.predicate == other.predicate and
                self.args == other.args and self.negated == other.negated)

    def __hash__(self):
        return hash((self.predicate, tuple(self.args), self.negated))


def parse_term(s):
    s = s.strip()
    if '(' not in s:
        return Term(s)
    name, rest = s.split('(', 1)
    args_str = rest[:-1]
    args = split_args(args_str)
    return Term(name.strip(), [parse_term(a) for a in args])

def split_args(s):
    args, depth, current = [], 0, []
    for char in s:
        if char == '(': depth += 1
        elif char == ')': depth -= 1
        if char == ',' and depth == 0:
            args.append(''.join(current))
            current = []
        else:
            current.append(char)
    if current:
        args.append(''.join(current))
    return args

def parse_literal(s):
    s = s.strip()
    negated = False
    if s.startswith('~') or s.startswith('¬'):
        negated = True
        s = s[1:].strip()
    name, rest = s.split('(', 1)
    args = [parse_term(a) for a in split_args(rest[:-1])]
    return Literal(name.strip(), args, negated)

def parse_clause(s):
    # Parses disjunctive clauses like "~food(x) | likes(John, x)"
    parts = [p.strip() for p in s.split('|')]
    return [parse_literal(p) for p in parts if p]


# --- 2. UNIFICATION ALGORITHM ---

def occur_check(var, x, theta):
    if var == x:
        return True
    elif isinstance(x, Term) and x.is_variable() and x in theta:
        return occur_check(var, theta[x], theta)
    elif isinstance(x, Term) and x.args:
        return any(occur_check(var, arg, theta) for arg in x.args)
    return False

def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    elif isinstance(x, Term) and x.is_variable() and x in theta:
        return unify(var, theta[x], theta)
    elif occur_check(var, x, theta):
        return None
    else:
        new_theta = theta.copy()
        new_theta[var] = x
        return new_theta

def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if theta is None:
        return None
    if x == y:
        return theta
    if isinstance(x, Term) and x.is_variable():
        return unify_var(x, y, theta)
    if isinstance(y, Term) and y.is_variable():
        return unify_var(y, x, theta)
    if isinstance(x, Term) and isinstance(y, Term) and x.args and y.args:
        if x.name != y.name or len(x.args) != len(y.args):
            return None
        for a, b in zip(x.args, y.args):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    if isinstance(x, list) and isinstance(y, list):
        if len(x) != len(y):
            return None
        for a, b in zip(x, y):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    return None

def substitute_term(term, theta):
    if term.is_variable():
        while term in theta:
            term = theta[term]
        return term
    if term.args:
        return Term(term.name, [substitute_term(a, theta) for a in term.args])
    return term

def substitute_literal(lit, theta):
    return Literal(lit.predicate, [substitute_term(a, theta) for a in lit.args], lit.negated)

def substitute_clause(clause, theta):
    # Remove duplicate literals after substitution
    res = []
    for lit in clause:
        sub_lit = substitute_literal(lit, theta)
        if sub_lit not in res:
            res.append(sub_lit)
    return res


# --- 3. STANDARDIZATION & RESOLUTION ENGINE ---

var_counter = 0
def standardize_variables(clause):
    global var_counter
    var_counter += 1
    mapping = {}

    def rename_term(t):
        if t.is_variable():
            if t not in mapping:
                mapping[t] = Term(f"{t.name}_{var_counter}")
            return mapping[t]
        if t.args:
            return Term(t.name, [rename_term(a) for a in t.args])
        return t

    return [Literal(l.predicate, [rename_term(a) for a in l.args], l.negated) for l in clause]

def format_clause(clause):
    if not clause:
        return "{ } (EMPTY CLAUSE)"
    return " | ".join(map(str, clause))

def resolve(ci, cj):
    resolvents = []
    for li in ci:
        for lj in cj:
            if li.predicate == lj.predicate and li.negated != lj.negated:
                theta = unify(li.args, lj.args, {})
                if theta is not None:
                    new_ci = [l for l in ci if l != li]
                    new_cj = [l for l in cj if l != lj]
                    combined = substitute_clause(new_ci + new_cj, theta)
                    resolvents.append((combined, theta))
    return resolvents


def print_proof_tree(records, node_id, prefix="", is_last=True, root=True):
    """Recursively print the resolution proof tree."""

    record = records[node_id - 1]
    clause = record["clause"]

    if root:
        print("\n" + "=" * 65)
        print("                 RESOLUTION PROOF TREE")
        print("=" * 65)
        print(f"[C{node_id}] {format_clause(clause)}")
    else:
        connector = "└── " if is_last else "├── "
        print(prefix + connector +
              f"[C{node_id}] {format_clause(clause)}")

    parents = record["parents"]

    if not parents:
        return

    next_prefix = prefix + ("    " if is_last else "│   ")

    if record["substitution"]:
        print(next_prefix + "    " +
              f"Substitution: {record['substitution']}")

    for i, parent_id in enumerate(parents):
        last = (i == len(parents) - 1)
        print_proof_tree(
            records,
            parent_id,
            next_prefix,
            last,
            False
        )


def format_substitution(theta):
    if not theta:
        return "None"

    return ", ".join(
        f"{key} = {substitute_term(value, theta)}"
        for key, value in theta.items()
    )


def fol_resolution(kb_strings, query_string):

    print("\n" + "=" * 65)
    print("          FIRST ORDER LOGIC RESOLUTION PROVER")
    print("=" * 65)

    # Each record stores the clause and its derivation history.
    records = []

    # STEP 1: INITIAL KNOWLEDGE BASE

    print("\nSTEP 1: INITIAL KNOWLEDGE BASE")
    print("-" * 65)

    for s in kb_strings:
        clause = parse_clause(s)

        records.append({
            "clause": clause,
            "parents": [],
            "substitution": None
        })

        print(
            f"[C{len(records)}] "
            f"{format_clause(clause)}"
        )

    # STEP 2: NEGATE QUERY

    print("\nSTEP 2: NEGATE THE QUERY")
    print("-" * 65)

    query = parse_literal(query_string)
    neg_query = [query.negate()]

    records.append({
        "clause": neg_query,
        "parents": [],
        "substitution": None
    })

    print(f"Query         : {query}")
    print(f"Negated query : {format_clause(neg_query)}")
    print(f"[C{len(records)}] Added to knowledge base")

    # STEP 3: RESOLUTION

    print("\nSTEP 3: RESOLUTION OPERATIONS")
    print("-" * 65)

    step = 1
    processed_pairs = set()

    while True:

        n = len(records)
        new_records = []

        for i in range(n):

            for j in range(i + 1, n):

                if (i, j) in processed_pairs:
                    continue

                processed_pairs.add((i, j))

                # Standardize variables apart.
                ci = standardize_variables(records[i]["clause"])
                cj = standardize_variables(records[j]["clause"])

                resolvents = resolve(ci, cj)

                for res, theta in resolvents:

                    parent1 = i + 1
                    parent2 = j + 1

                    substitution = format_substitution(theta)

                    print(f"\nStep {step}")
                    print(
                        f"Resolve C{parent1} and C{parent2}"
                    )
                    print(
                        f"   Parent 1 : "
                        f"{format_clause(records[i]['clause'])}"
                    )
                    print(
                        f"   Parent 2 : "
                        f"{format_clause(records[j]['clause'])}"
                    )
                    print(f"   MGU      : {substitution}")
                    print(f"   Result   : {format_clause(res)}")

                    step += 1

                    # Empty clause means contradiction.
                    if not res:

                        records.append({
                            "clause": [],
                            "parents": [parent1, parent2],
                            "substitution": substitution
                        })

                        empty_id = len(records)

                        print("\n" + "=" * 65)
                        print("CONTRADICTION FOUND!")
                        print("=" * 65)

                        print(
                            f"Empty clause derived: C{empty_id}"
                        )

                        print_proof_tree(records, empty_id)

                        print("\nCONCLUSION:")
                        print(f"{query} IS PROVED TRUE.")
                        print("=" * 65)

                        return True

                    # Avoid duplicate clauses.
                    already_exists = any(
                        res == record["clause"]
                        for record in records + new_records
                    )

                    if not already_exists:

                        new_records.append({
                            "clause": res,
                            "parents": [parent1, parent2],
                            "substitution": substitution
                        })

        if not new_records:

            print("\n" + "=" * 65)
            print("NO NEW CLAUSES GENERATED.")
            print("Unable to prove the given query.")
            print("=" * 65)

            return False

        # Add newly derived clauses.
        for record in new_records:

            records.append(record)

            print(
                f"\nAdded C{len(records)}: "
                f"{format_clause(record['clause'])}"
            )



# --- 4. INTERACTIVE USER INPUT ---

if __name__ == "__main__":
    print("=== FIRST-ORDER LOGIC RESOLUTION PROVER ===")
    print("Syntax Rules:")
    print("  - Variables: lowercase (e.g., x, y, z)")
    print("  - Constants: Capitalized (e.g., John, Peanuts, Anil)")
    print("  - Negation : ~ (e.g., ~food(x))")
    print("  - OR       : | (e.g., ~food(x) | likes(John, x))")
    print("-" * 50)

    choice = input("Press [1] to run the Lab Case Study (John & Peanuts) or [2] to enter custom input: ").strip()

    if choice == "1":
        # Exact CNF clauses from Slide 10 & 11 of your lab PDF
        kb = [
            "~food(x) | likes(John, x)",
            "food(Apple)",
            "food(vegetables)",
            "~eats(y, z) | killed(y) | food(z)",
            "eats(Anil, Peanuts)",
            "alive(Anil)",
            "~eats(Anil, w) | eats(Harry, w)",
            "killed(g) | alive(g)",
            "~alive(k) | ~killed(k)"
        ]
        query = "likes(John, Peanuts)"
        fol_resolution(kb, query)
    else:
        num = int(input("\nEnter number of CNF clauses in Knowledge Base: "))
        kb = []
        for i in range(num):
            clause_input = input(f"Clause {i+1}: ")
            kb.append(clause_input)
        query = input("\nEnter Query to prove (e.g., likes(John, Peanuts)): ")
        fol_resolution(kb, query)


=== FIRST-ORDER LOGIC RESOLUTION PROVER ===
Syntax Rules:
  - Variables: lowercase (e.g., x, y, z)
  - Constants: Capitalized (e.g., John, Peanuts, Anil)
  - Negation : ~ (e.g., ~food(x))
  - OR       : | (e.g., ~food(x) | likes(John, x))
--------------------------------------------------
Press [1] to run the Lab Case Study (John & Peanuts) or [2] to enter custom input: 2

Enter number of CNF clauses in Knowledge Base: 4
Clause 1: ~Allergy(x) | Sneezes(x)
Clause 2: ~LivesWith(x,y) | ~Cat(y) | ~AllergicTo(x,y) | Allergy(x)
Clause 3: Cat(Tom)
Clause 4: AllergicTo(Mary,Tom)

Enter Query to prove (e.g., likes(John, Peanuts)): Sneezes(Mary)

          FIRST ORDER LOGIC RESOLUTION PROVER

STEP 1: INITIAL KNOWLEDGE BASE
-----------------------------------------------------------------
[C1] ~Allergy(x) | Sneezes(x)
[C2] ~LivesWith(x, y) | ~Cat(y) | ~AllergicTo(x, y) | Allergy(x)
[C3] Cat(Tom)
[C4] AllergicTo(Mary, Tom)

STEP 2: NEGATE THE QUERY
-------------------------------------------------